In [ ]:
#========================================================================
#
# Name: StepInSSim.ipynb
#
# Date: 11/01/2026
#
# Author: MT
# 
# Description: Simulated data to demonstrate step-in modelling.
#
# Steps:
# Step 0: Imports
# # Step 1: Simulate Data
# Step 2: Modelling
# Step 3: SAS Code (reference)
#
#========================================================================

# Step-in Phased Logistic Regression

## Background
This notebook demonstrates step-in phased logistic regression, which results in down-weighting influence of dominant features while up-weighting the influence of less dominant features. 

## Take-aways
- The method can help elevate the impact of less dominant features in the presence of more dominant features.
- The method involves fitting models in phases, connected via an offset where the first phase models the less dominant features
- The result is a "worse" model (lower Gini and model fit) but with more balanced feature influence.

## Approach
This is illustrated using a simulated dataset containing two predictor variables:
- `x1` - a continuous less dominant feature (true population parameter 1.0)
- `payments_missed` - a more dominant binary feature (true population parameter 3.0)
- `y` - target variable constructed using a logistic function of `x1` and `payments_missed`

## Details
### Steps:
1. **Data Simulation**: Generate a synthetic dataset with specified characteristics.
2. **Overall Logistic Regression**: Fit a full logistic regression model using both `x1` and `payments_missed`. This serves as a benchmark as the dominant feature will likely overshadow the less dominant feature.
3. **Phase 1 Modeling**: Fit a logistic regression model using `x1` as the predictor.
4. **Phase 2 Modeling**: Fit a logistic regression model using `payments_missed` as the predictor, with the linear predictor from Phase 1 as an offset.

### Mathematical Formulation:

Let $Y_i=1$ if the event occurs for observation $i$, and $Y_i=0$ otherwise. Let further $E[Y_i=1]=\mu_i$. With $g^{-1}(\mu_i)=\text{logit}(\mu_i)$ is the canonical link function, the logistic regression model estimates the probability of the event as follows:

- **Full Model**:
$$
g^{-1}(\mu_i) = \gamma_0 + \gamma_1 x_1 + \gamma_2\,\text{payments\_missed}
$$

- **Phase 1 Model**:

To elevate the influence of `x1`, we first fit a model using only `x1`:
$$
g^{-1}(\mu_{1i}) = \alpha_0 + \alpha_1 x_1 
$$

The fitted model is then used to compute the linear predictor (log-odds) for each observation:
$$
\hat{\eta}_{1i} = g^{-1}(\hat{\mu}_{1i}) = \hat{\alpha}_0 + \hat{\alpha}_1 x_{1i}
$$

- **Phase 2 Model**:

Next, we fit a second model using `payments_missed` as the predictor, incorporating the linear predictor from Phase 1 as an offset:
$$
g^{-1}(\mu_{2i}) = \beta_0 + \beta_1\,\text{payments\_missed} + g^{-1}(\hat{\mu}_{1i}) 
$$

This is equivalent to:
$$
g^{-1}(\mu_{2i}) - g^{-1}(\hat{\mu}_{1i}) = \beta_0 + \beta_1\,\text{payments\_missed}
$$

In this way, the phase 2 model here is akin to a residual model, where the effect of `x1` is preserved from Phase 1, and the effect of `payments_missed` is estimated in Phase 2. In fact, if ordinary least squares was instead selected as the modeling approach along with an identity link function, this would be equivalent to fitting a model to the residuals from Phase 1:

$$
y_i - \hat{y}_i = \beta_0 + \beta_1\,\text{payments\_missed}
$$

- **Final Model**:

The final fitted model combines the effects of both predictors, with `x1`'s influence preserved from Phase 1 and `payments_missed`'s influence added in Phase 2.

$$
g^{-1}(\hat{\mu}_{2i}) = \hat{\beta}_0 + \hat{\beta}_1\,\text{payments\_missed} + \hat{\alpha}_0 + \hat{\alpha}_1 x_{1i}
$$

In [77]:
#========================================================================
# Step 0: Imports

# Numpy, Pandas and copy
import numpy as np
import pandas as pd
import copy

# StatsModels for Logistic Regression
import statsmodels.formula.api as smf

# sklearn for AUC / Gini
from sklearn.metrics import roc_auc_score
#========================================================================

In [88]:
#========================================================================
# # Step 1: Simulate Data

# Set seed for reproducibility
np.random.seed(0)

# Sample size
n = 100_000

# Correlation matrix for 2-dim normal:
rho = 0.7
R = np.array([
    [1.00, rho,  ],
    [rho,  1.00, ],
])

# Draw correlated standard normals
Z = np.random.normal(size=(n, 2))

# Cholesky decomposition
L = np.linalg.cholesky(R)

# Matrix multiplication ZL^T
X = Z @ L.T

# DataFrame
df = pd.DataFrame(X, columns=["x1",  "Latent"])

# Slice pm_latent into payments_missed using quantiles for desired proportions
p = np.array([0.7, 0.3]) # 70% with 0 up-to-date, 30% with 1 missed
cuts = np.quantile(df["Latent"], np.cumsum(p)[:-1])   
df['payments_missed'] = np.digitize(df["Latent"], bins=cuts) 
df.drop(columns=["Latent"], inplace=True)

# Generate score as linear combination of x1 and payments_missed plus noise
score = (
    -7.5
    + 1 * df["x1"]
    + 3 * df["payments_missed"]
    + np.random.normal(0, 2, n) # noise
)

# Generate binary target y using logistic model
prob = 1 / (1 + np.exp(-score))
y = np.random.binomial(1, prob, n)
df["y"] = y

# Save to csv
df=df.round(6)
df.to_csv("StepInSim_data2.csv", index=False)

# Review default rate by payments_missed
temp=pd.crosstab(df["payments_missed"], df["y"], margins=True)
temp['default_rate'] = temp[1] / temp['All']
print('Crosstab - Payments Missed vs y\n', temp)
#========================================================================

Crosstab - Payments Missed vs y
 y                    0     1     All  default_rate
payments_missed                                   
0                69718   282   70000      0.004029
1                27110  2890   30000      0.096333
All              96828  3172  100000      0.031720


In [92]:
#========================================================================
# Step 2: Modelling

str='###########################################################'
str2='~~~~~~~~~'

# Full model
print(str)
print(str2, 'FULL MODEL')
m = smf.logit("y ~ x1 + payments_missed", data=df).fit()
gini=roc_auc_score(df["y"], m.predict(df))*2-1
print(m.summary())
print('Gini: ', np.round(gini,4))
print(str, '\n')

# Step-in modelling - Phase 1
print(str)
print(str2, 'PHASE1 MODEL - x1 Only')
m1 = smf.logit("y ~ x1", data=df).fit()
gini=roc_auc_score(df["y"], m1.predict(df))*2-1
eta1=m1.predict(df, which="linear")
print(m1.summary())
print('Gini: ', np.round(gini,4))
print(str, '\n')

# Step-in modelling - Phase 2
print(str)
print(str2, 'PHASE2 MODEL - payments_missed with Phase1 offset')
m2 = smf.logit("y ~ payments_missed", data=df, offset=eta1).fit()
gini=roc_auc_score(df["y"], m2.predict(df, which="linear")+eta1)*2-1
print(m2.summary())
print('Gini: ', np.round(gini,4))
print(str, '\n')
#========================================================================

###########################################################
~~~~~~~~~ FULL MODEL
Optimization terminated successfully.
         Current function value: 0.108408
         Iterations 10
                           Logit Regression Results                           
Dep. Variable:                      y   No. Observations:               100000
Model:                          Logit   Df Residuals:                    99997
Method:                           MLE   Df Model:                            2
Date:                Sun, 11 Jan 2026   Pseudo R-squ.:                  0.2294
Time:                        17:28:00   Log-Likelihood:                -10841.
converged:                       True   LL-Null:                       -14067.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept       